# EEG Signal Processing Toolbox - DSP Engine Demonstration

This notebook serves as an interactive demonstration of the digital signal processing (DSP) architecture powering the **EEG Toolbox**. The pipeline demonstrates raw physiological data extraction, artifact removal via infinite impulse response (IIR) notch filtering, and a comparative mathematical analysis of finite impulse response (FIR) filter designs applied to specific neural frequency bands.

### 1. Environment Setup & Data Loading
First, we initialize the environment and load a standard European Data Format (`.edf`) file from the sample repository.

In [ ]:
import os
import matplotlib.pyplot as plt
import pyedflib
import data_loader
import data_displayer as disp
import signal_processing
import montages 
import numpy as np

sample_dir = "EEG_Samples"
files = [f for f in os.listdir(sample_dir) if f.endswith('.edf')]

print(file_names := f"EEG files available: {files}")

if files:
    file_path = os.path.join(sample_dir, files[9])
    f_edf = pyedflib.EdfReader(file_path)
    print(f"EEG signal '{files[9]}' loaded succesfully")

### 2. Signal Extraction and Metadata Parsing
With the `.edf` file loaded, we extract the core data matrix and its associated metadata, including the anatomical channel labels, sampling frequency ($f_s$), and total recording duration.

NOTE: We are currently adding noise to "Afz" channel to later show bad channel removal & interpolation.

In [ ]:
data_matrix, n_channels, channel_names, freq, time, dimension_unit = data_loader.load_edf_data(file_path)

# --- Afz channel noise generation ---
bad_ch_idx = 26
bad_channel = channel_names[bad_ch_idx]
std_channel = np.std(data_matrix[bad_ch_idx, :])
noise_scale = std_channel*3
white_noise = np.random.normal(loc=0.0, scale=noise_scale, size=data_matrix.shape[1])
data_matrix[bad_ch_idx, :] += white_noise

print(f"Found {n_channels} available channels: " + f"{channel_names}")
print(f"Sampling frequency: {freq} Hz")
print(f"Sampling time: {time} s")


### 3. Electrode Layout Plotting
After loading the data, we scan the channels to get the electrode layout and proceed plotting the scalp with the electrodes to better visualize the position of each source.

In [ ]:
electrode_layout = montages.get_layout(n_channels)
fig_topomap = disp.scalp_map(electrode_layout)
plt.show(fig_topomap)

### 4. Unfiltered Signal Exploration (Time Domain)
To visually evaluate the filtering pipeline, we plot the EEG channels divided by lobe. We select a time window (from 1 to 20 seconds long). Different colors have been assigned to distinguish the channels. The raw signal is plotted below to highlight baseline wander and high-frequency artifacts characteristic of unconditioned neurophysiological recordings.

In [ ]:
time_start = 0
time_width = 10
fig_original = disp.displayer(data_matrix, 
                              time_start, 
                              time_width, 
                              channel_names,
                              freq,
                              dimension_unit
                              )

plt.show(fig_original)

### 5. Bad Channel Removal & Interpolation
After plotting the signal, we proceed to apply the processing pipeline. The first option selectable is "Bad Channel Remove & Interpolate", which sets to zero noisy or absent channels to interpolate them weighting all the channels in the scalp. Single channel plotting is presented to inspect singularly all the electrodes. As we could see from the previous plot, channel "Afz" is damaged. We set it to zero and then recreate it from the rest of the signal. 

In [ ]:

fig_raw = disp.display_data(data_matrix[bad_ch_idx, :], 
                  bad_channel, 
                  time, 
                  freq, 
                  dimension_unit
                  )
print(f"EEG Channel {bad_channel} - Raw")
plt.show(fig_raw)

data_matrix = signal_processing.Remove_bad_channels(data_matrix, [bad_ch_idx])
data_matrix = signal_processing.Interpolate_bad_channels(data_matrix, channel_names, [bad_ch_idx], electrode_layout)
fig_new = disp.display_data(data_matrix[bad_ch_idx, :], 
                  bad_channel, 
                  time, 
                  freq, 
                  dimension_unit
                  )
print(f"EEG Channel {bad_channel} - Interpolated")
plt.show(fig_new)


### 6. Downsampling
EEG signals are sometimes sampled at higher frequencies to achieve a better temporal resolution. However, data that have been sampled at higher frequencies are heavy to process, specifically on a personal computer. For this reason, downsampling the signal is a solution to make the analysis more efficient. Our sample has been sampled to 160 Hz, so we only show the function by cutting down the frequency drastically, but we will keep working with the original signal.

In [ ]:
downsample_factor = 10
data_downsampled = data_matrix.copy()
freq_downsampled = freq/downsample_factor
data_downsampled = signal_processing.Downsample(data_downsampled,
                                                downsample_factor)

fig_downsampled = disp.displayer(data_downsampled,
                                 time_start,
                                 time_width,
                                 channel_names,
                                 freq_downsampled,
                                 dimension_unit
                                 )
plt.show(fig_downsampled)

### 7. Current Artifact Removal
A common source of contamination in EEG recordings is electromagnetic interference from the power grid, as it's visible from the image. We apply an IIR Notch Filter centered at $60\text{ Hz}$ using a high quality factor ($Q = 400$). This ensures a highly specific attenuation of the powerline artifact while strictly preserving the integrity and power of the adjacent neural frequency bands.

In [ ]:
print("Removing current artifact")

data_matrix = signal_processing.Current_Remover(data_matrix, freq, f_remove=60.0, Q=400.0)

fig_clean = disp.displayer(data_matrix,
                           time_start,
                           time_width,
                           channel_names,
                           freq, 
                           dimension_unit
                           )

plt.show(fig_clean)

### 8. Re-Referencing 
The EEG reference can be changed using **Common Average Reference**: we subtract the average of the whole scalp from each channel, to correct starting bias and offsets.

In [ ]:
data_matrix = signal_processing.Common_average_reference(data_matrix)

fig_rereferenced = disp.displayer(data_matrix,
                                 time_start,
                                 time_width,
                                 channel_names,
                                 freq,
                                 dimension_unit
                                 )
plt.show(fig_rereferenced)

### 9. Independent Component Analysis
The EEG signal is heavily contaminated by non-neural electrical activity such as ocular artifacts (eye-blinks), myogenic activity (muscle contractions), and cardiac interference (ECG). To isolate cortical oscillations, **Independent Component Analysis (ICA)** assumes a generative linear model $x = As$, where the observed multichannel signals $x$ are linear mixtures of statistically independent sources $s$. 

Using the FastICA algorithm, the toolbox estimates an unmixing matrix $W$ to recover the independent components ($s = Wx$) by maximizing their statistical non-Gaussianity. Subsequently, the Power Spectral Density (PSD) of each isolated component is computed via Welch's method. The components are displayed in both the time domain and frequency domain, allowing the user to visually identify and reject artifactual sources before mathematically reconstructing the clean EEG signal.

In [ ]:
n_components = 5

S_, A_, data_mean = signal_processing.Compute_ICA(data_matrix,
                                         n_components
                                        )
freq_PSD_components, PSD_components = signal_processing.Compute_PSD(S_,
                                                         freq,
                                                        )
fig_components = disp.show_components(S_,
                                      freq_PSD_components,
                                      PSD_components,
                                      freq,
                                      dimension_unit
                                      )
                                     

We recognize component 3 to be probable noise due to the random peaks at 40s, 75s and around 120s. Component 5 is the ECG signal, due to the periodical downpeaks. We remove the two selected components to analyze the independent EEG signal.

In [ ]:
bad_components = [3, 5]

data_matrix = signal_processing.Reconstruct_from_ICA(S_,
                                                     A_,
                                                     data_mean,
                                                     bad_components)

fig_reconstructed_data = disp.displayer(data_matrix,
                                        time_start,
                                        time_width,
                                        channel_names,
                                        freq,
                                        dimension_unit
                                        )

### 10. FIR Filter Design and Comparative Analysis
Next, we design a bandpass filter targeting the **Theta ($\theta$) band (3-7 Hz)**, which is often associated with cognitive processing and sleep states. 

To ensure strict linear phase (constant group delay) and minimize signal distortion, we evaluate three distinct FIR filter design algorithms:
*   **Window Method** (using a Hanning window)
*   **Equiripple** (Parks-McClellan algorithm for minimax error)
*   **Least Squares** (minimizing the integral of the squared error)

The filters are evaluated with identical transition widths ($0.2\text{ Hz}$) and a high tap order ($N = 1001$) to visually assess the engineering trade-offs between transition steepness, stopband attenuation, and passband ripple in the same starting conditions.

In [ ]:
trans_width = 0.2
order = int(1e3)
f_cut_l = 3.0
f_cut_h = 7.0
methods = ["Window Method", "Equiripple", "Least Squares"]
filter_selected = "Bandpass Filter"

print("Filtering in \u03B8 band (3-7 Hz) using Hamming Window, Equiripple, Least Sqares")
print("Parameters have been equally set")
print(f"Transition Width: {trans_width} Hz")
print(f"Order: {order+1} Taps")

Fir_window = signal_processing.Fir_designer(freq, order, filter_selected, methods[0], f_cut_l, f_cut_h, trans_width, window_selected="Hanning", beta = None)
Fir_equiripple = signal_processing.Fir_designer(freq, order, filter_selected, methods[1], f_cut_l, f_cut_h, trans_width, window_selected=None, beta=None)
Fir_lsquares = signal_processing.Fir_designer(freq, order, filter_selected, methods[2], f_cut_l, f_cut_h, trans_width, window_selected=None, beta=None)

print("\nComparing All FIR Filters")

fig_Fir_compare = disp.compare_filter(Fir_window, Fir_equiripple, Fir_lsquares, filter_selected, freq)
plt.show(fig_Fir_compare)


### 11. Application of the Optimal Filter
Based on the comparative Bode analysis above, we select the **Least Squares** FIR design for its optimal balance of attenuation and passband flatness. In fact, the **Least Squares** option provides a sharp transition and a good attenuation compared to the ripple factor outside the window. We convolve our notch-filtered EEG data with the selected FIR taps to safely isolate the $\theta$-band oscillations.

In [ ]:
method_selected = methods[2]
data_filtered = signal_processing.Filter_data(data_matrix,
                                              Fir_lsquares)

fig_data_filtered = disp.displayer(data_filtered,
                                   time_start,
                                   time_width,
                                   channel_names,
                                   freq,
                                   dimension_unit
                                   )
                                    
print(f"Method selected: {method_selected}")
plt.show(fig_data_filtered)

### 12. Power Spectral Density
We finally plot the Power Spectral Density (PSD) of the data to highlight the contribution of each lobe in the $\theta$-band.

In [ ]:
freq_PSD, PSD_channels = signal_processing.Compute_PSD(data_filtered,
                                                       freq
                                                       )

fig_PSD = disp.display_PSD(freq_PSD,
                           PSD_channels,
                           channel_names
                           )

### Conclusion
We have successfully visualized the $\theta$-band activity of the EEG signal, free from non-brain activity and current artifacts.